# Demo data for the front end

Precomputes everything the live demo needs for the 10 student-test trips, so the website does no model work on the spot.

Uses the saved **tuned XGBoost ranker** predictions (`student_test_predictions.csv`, the model chosen in `results/final_selection.json`). Per trip it saves the GPS pings, the predicted and reference road segment for each ping, the road geometry involved, the reference route, and trip metrics. Output: `frontend/data.js`.

In [1]:
%run 06_model_helpers.ipynb
%run 08_hmm_helpers.ipynb

selection = json.loads(Path("results/final_selection.json").read_text())
prediction_file = Path(selection["model_dir"]) / "student_test_predictions.csv"
predictions = pd.read_csv(prediction_file)
test_points = pd.read_csv("outputs/student_test/points.csv", dtype={"trip_id": str})
print("Model:", selection["model_path"])
print("Prediction rows:", f"{len(predictions):,}", "| trips:", test_points["trip_id"].nunique())

Model: results/step7_xgb/tuned_sample10000/model.json
Prediction rows: 1,186,138 | trips: 10


## Top-1 road for every GPS ping

Highest score wins; ties go to the lowest segment ID, as in the shared scoring helper.

In [2]:
best = predictions.sort_values(["group_id", "score", "segment_id"], ascending=[True, False, True]).drop_duplicates("group_id")
best = best.set_index("group_id")
test_points["pred_arc"] = test_points["group_id"].map(best["segment_id"]).fillna(-1).astype(int)
test_points["confidence"] = test_points["group_id"].map(best["score"]).fillna(0)
test_points["correct"] = test_points["pred_arc"] == test_points["true_arc_id"]
print(test_points.groupby("trip_id")["correct"].mean().round(4))

trip_id
00000000    0.9573
00000007    0.9180
00000009    0.8904
00000040    0.9359
00000052    0.9618
00000056    0.9149
00000065    0.8906
00000076    0.9300
00000077    0.7614
00000091    0.9016
Name: correct, dtype: float64


## Per-trip data and metrics

Metrics follow the shared rules: valid (non-flagged) points, one winner per point, candidate misses stay in the Top-1 denominator. Route continuity counts adjacent predicted segments that are connected on the road graph.

In [3]:
def round_list(values, digits):
    return [round(float(v), digits) for v in values]

trips = []
for trip_id, pts in test_points.groupby("trip_id", sort=True):
    pts = pts.sort_values("point_id")
    folder = Path("Student folder/student_test") / trip_id
    nodes = pd.read_csv(folder / f"{trip_id}.nodes", sep=r"\s+", header=None).to_numpy()
    arcs = pd.read_csv(folder / f"{trip_id}.arcs", sep=r"\s+", header=None).to_numpy()
    route = pd.read_csv(folder / f"{trip_id}.route", sep=r"\s+", header=None)[0].to_numpy()

    # Metrics on valid points, using the shared scoring helper.
    valid = pts[~pts["bad_gps"]]
    rows = predictions[predictions["group_id"].isin(valid["group_id"])].copy()
    rows["label"] = (rows["segment_id"] == rows["group_id"].map(valid.set_index("group_id")["true_arc_id"])).astype("int8")
    metrics = score_points(rows, rows["score"].to_numpy(), valid)

    # Route continuity of the predicted sequence.
    network = load_network(trip_id, partition="student_test")
    chosen = pts["pred_arc"].to_numpy()
    pairs = [(a, b) for a, b in zip(chosen[:-1], chosen[1:]) if min(a, b) >= 0]
    reachable = sum(transition_is_reachable(network, int(a), int(b)) for a, b in pairs)

    # Road geometry: every candidate segment plus the reference route.
    candidate_ids = np.union1d(predictions.loc[predictions["group_id"].isin(pts["group_id"]), "segment_id"].unique(), route)
    segments = {int(i): round_list(np.r_[nodes[arcs[i, 0]][::-1], nodes[arcs[i, 1]][::-1]], 5) for i in candidate_ids}

    trips.append({
        "id": trip_id,
        "points": len(pts), "valid_points": len(valid), "flagged_points": int(pts["bad_gps"].sum()),
        "duration_s": float(pts["time"].max() - pts["time"].min()),
        "top1": metrics["top1"], "macro_f1": metrics["macro_f1"], "log_loss": metrics["log_loss"],
        "candidate_coverage": metrics["candidate_coverage"], "confusion_matrix": metrics["confusion_matrix"],
        "route_continuity": reachable / max(len(pairs), 1), "disconnected_jumps": len(pairs) - reachable,
        "lat": round_list(pts["lat"], 6), "lon": round_list(pts["lon"], 6), "t": round_list(pts["time"], 1),
        "pred": pts["pred_arc"].astype(int).tolist(), "truth": pts["true_arc_id"].astype(int).tolist(),
        "conf": round_list(pts["confidence"], 3), "bad": pts["bad_gps"].astype(int).tolist(),
        "route": route.astype(int).tolist(), "segments": segments,
    })
    print(trip_id, f"Top-1 {metrics['top1']:.4f}", f"continuity {reachable / max(len(pairs), 1):.4f}", f"segments {len(segments):,}")

00000000 Top-1 0.9573 continuity 0.9984 segments 1,893


00000007 Top-1 0.9180 continuity 1.0000 segments 721


00000009 Top-1 0.8904 continuity 1.0000 segments 3,354


00000040 Top-1 0.9359 continuity 1.0000 segments 2,302


00000052 Top-1 0.9618 continuity 1.0000 segments 792


00000056 Top-1 0.9149 continuity 0.9987 segments 5,029
00000065 Top-1 0.8906 continuity 1.0000 segments 2,233


00000076 Top-1 0.9300 continuity 1.0000 segments 5,805


00000077 Top-1 0.7614 continuity 1.0000 segments 3,060
00000091 Top-1 0.9016 continuity 1.0000 segments 2,061


## Check against the saved overall result, then write `frontend/data.js`

The pooled Top-1 across the 10 trips must match `results/final_selection.json`. `data.js` is a plain script (not JSON) so `index.html` also works by double-click, with no server.

In [4]:
overall = selection["metrics"]
pooled_top1 = sum(t["top1"] * t["valid_points"] for t in trips) / sum(t["valid_points"] for t in trips)
assert abs(pooled_top1 - overall["top1"]) < 1e-9, (pooled_top1, overall["top1"])

demo = {
    "model": "XGBoost ranker (tuned, Bayesian search, classifier only - no HMM)",
    "overall": {k: overall[k] for k in ["top1", "macro_f1", "log_loss", "route_continuity", "valid_points",
                                        "disconnected_jumps", "prediction_ms_per_trip"]},
    "trips": trips,
}
out = Path("frontend/data.js")
out.write_text("window.DEMO = " + json.dumps(demo, separators=(",", ":"), allow_nan=False) + ";\n")
print(f"Pooled Top-1 {pooled_top1:.4f} matches saved result | wrote {out} ({out.stat().st_size / 1e6:.1f} MB)")

Pooled Top-1 0.8915 matches saved result | wrote frontend/data.js (2.1 MB)
